# Laboratório — Losses de regressão e classificação binária

Implementaremos MSE e binary cross-entropy (BCE) estável em **NumPy puro**.
O notebook mantém previsões, logits e probabilidades como objetos distintos, rejeita
broadcasting silencioso e testa extremos numéricos. Não há backpropagation, autograd,
PyTorch, TensorFlow ou JAX.

Este é o artefato executável da Aula 06 do M5 — Redes Neurais do Zero.


## Goal

Ao final, teremos evidência executável de que:

1. o MSE manual e a implementação coincidem;
2. `none`, `sum`, `mean` e média ponderada têm contratos explícitos;
3. shapes incompatíveis são rejeitados antes do broadcasting;
4. a BCE estável coincide com a fórmula em probabilidades em logits moderados;
5. a BCE baseada em logits permanece finita para valores extremos;
6. clipping de probabilidades altera a penalidade;
7. médias globais são invariantes ao particionamento quando soma e contagem são preservadas.


## Setup

- Python >= 3.11
- NumPy >= 1.26
- Matplotlib >= 3.8
- nbformat >= 5.9 apenas para validar o arquivo

Dados: vetores explícitos e curvas sintéticas determinísticas. Não há download,
credencial, dataset externo ou estado oculto. Seed: `20260909`; dtype: `float64`.


In [ ]:
import platform

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260909
DTYPE = np.float64
rng = np.random.default_rng(SEED)

print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "matplotlib": plt.matplotlib.__version__,
    "seed": SEED,
    "dtype": str(np.dtype(DTYPE)),
})


## Steps

### 1. Validadores e redução

Nesta aula, previsão/logit e alvo devem ter exatamente o mesmo shape. Aceitar
`(m, 1)` e `(m,)` produziria, por broadcasting, uma matriz `(m, m)` semanticamente errada.


In [ ]:
class LossContractError(ValueError):
    pass


def same_shape(a, b, names=("prediction", "target")):
    a = np.asarray(a, dtype=DTYPE)
    b = np.asarray(b, dtype=DTYPE)
    if a.shape != b.shape:
        raise LossContractError(
            f"{names[0]} e {names[1]} devem ter o mesmo shape; "
            f"recebidos {a.shape} e {b.shape}"
        )
    if a.size == 0:
        raise LossContractError("a loss exige pelo menos um elemento")
    if not (np.isfinite(a).all() and np.isfinite(b).all()):
        raise FloatingPointError("entradas da loss devem ser finitas")
    return a, b


def reduce_loss(values, reduction="mean"):
    values = np.asarray(values, dtype=DTYPE)
    if values.size == 0 or not np.isfinite(values).all():
        raise FloatingPointError("loss elementar vazia ou não finita")
    if reduction == "none":
        return values.copy()
    if reduction == "sum":
        return float(np.sum(values, dtype=DTYPE))
    if reduction == "mean":
        return float(np.mean(values, dtype=DTYPE))
    raise LossContractError("reduction deve ser 'none', 'sum' ou 'mean'")


### 2. MSE: implementação e exemplo manual

Usamos

$$
\operatorname{MSE}=\frac{1}{N}\sum_{j=1}^{N}(\hat y_j-y_j)^2,
$$

em que $N$ é o número total de elementos reduzidos, não apenas o número de linhas.


In [ ]:
def mse(prediction, target, reduction="mean"):
    prediction, target = same_shape(prediction, target)
    elementwise = np.square(prediction - target)
    return reduce_loss(elementwise, reduction)


y_reg = np.array([1.0, 2.0, -1.0])
pred_reg = np.array([2.0, 0.0, -1.0])

residuals = pred_reg - y_reg
squared = mse(pred_reg, y_reg, reduction="none")
mse_value = mse(pred_reg, y_reg)

print("resíduos:", residuals)
print("quadrados:", squared)
print(f"soma={squared.sum():.6f}; MSE={mse_value:.6f}")


### 3. Outlier e unidade da penalidade

O erro quadrático cresce mais rápido que o erro absoluto. A comparação com MAE é
diagnóstica: não implementaremos outra loss de treinamento nesta aula.


In [ ]:
errors_typical = np.array([1.0, -1.0, 1.0, -1.0])
errors_with_outlier = np.append(errors_typical, 10.0)

summary_outlier = {
    "MSE_sem_outlier": float(np.mean(errors_typical**2)),
    "MSE_com_outlier": float(np.mean(errors_with_outlier**2)),
    "MAE_sem_outlier": float(np.mean(np.abs(errors_typical))),
    "MAE_com_outlier": float(np.mean(np.abs(errors_with_outlier))),
}
print(summary_outlier)


### 4. Saídas múltiplas e denominador

Uma matriz `(m, d_out)` contém `m * d_out` elementos. `mean` reduz todos eles;
`sum` preserva o numerador para agregação posterior.


In [ ]:
y_multi = np.array([[1.0, 10.0], [3.0, 20.0], [5.0, 30.0]])
pred_multi = np.array([[2.0, 8.0], [3.0, 23.0], [1.0, 30.0]])
elementwise_multi = mse(pred_multi, y_multi, "none")
sum_multi = mse(pred_multi, y_multi, "sum")
mean_multi = mse(pred_multi, y_multi, "mean")

print("loss elementar:\n", elementwise_multi)
print({"N": elementwise_multi.size, "sum": sum_multi, "mean": mean_multi})


### 5. Sigmoid estável e BCE em probabilidades

A forma em probabilidades serve como referência apenas em logits moderados. A sigmoid
usa dois ramos para nunca formar `exp(1000)`.


In [ ]:
def sigmoid_stable(z):
    z = np.asarray(z, dtype=DTYPE)
    out = np.empty_like(z)
    positive = z >= 0.0
    out[positive] = 1.0 / (1.0 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    out[~positive] = exp_z / (1.0 + exp_z)
    return out


def bce_from_probabilities(probability, target, reduction="mean"):
    probability, target = same_shape(
        probability, target, names=("probability", "target")
    )
    if np.any((probability <= 0.0) | (probability >= 1.0)):
        raise LossContractError("probabilidades devem estar estritamente entre 0 e 1")
    if np.any((target < 0.0) | (target > 1.0)):
        raise LossContractError("alvos BCE devem estar em [0, 1]")
    values = -(target * np.log(probability) +
               (1.0 - target) * np.log1p(-probability))
    return reduce_loss(values, reduction)


### 6. BCE estável diretamente dos logits

Implementamos

$$
\ell(y,z)=\log(1+e^z)-yz
=\operatorname{logaddexp}(0,z)-yz.
$$

`np.logaddexp` preserva a identidade sem exponenciar um número positivo extremo.


In [ ]:
def bce_with_logits(logits, target, reduction="mean"):
    logits, target = same_shape(logits, target, names=("logits", "target"))
    if np.any((target < 0.0) | (target > 1.0)):
        raise LossContractError("alvos BCE devem estar em [0, 1]")
    elementwise = np.logaddexp(0.0, logits) - target * logits
    if not np.isfinite(elementwise).all():
        raise FloatingPointError("BCE produziu valor não finito")
    return reduce_loss(elementwise, reduction)


logits_manual = np.array([0.0, 2.0, -2.0])
y_manual = np.array([0.0, 1.0, 0.0])
bce_manual = bce_with_logits(logits_manual, y_manual, "none")

print("BCE elementar:", np.round(bce_manual, 9))
print(f"BCE média={bce_manual.mean():.9f}")


### 7. Equivalência no domínio moderado

Quando a sigmoid ainda produz probabilidades estritamente entre zero e um, as duas
fórmulas devem coincidir até o erro de arredondamento.


In [ ]:
moderate_logits = np.linspace(-12.0, 12.0, 401)
moderate_targets = rng.integers(0, 2, size=moderate_logits.shape).astype(DTYPE)
moderate_probabilities = sigmoid_stable(moderate_logits)

reference = bce_from_probabilities(
    moderate_probabilities, moderate_targets, reduction="none"
)
stable = bce_with_logits(moderate_logits, moderate_targets, reduction="none")
equivalence_error = float(np.max(np.abs(reference - stable)))

print(f"erro máximo entre as formas={equivalence_error:.3e}")


### 8. Extremos: onde a forma ingênua quebra

Testamos previsões corretas e incorretas com logits de magnitude 1000. A forma estável
deve produzir aproximadamente `[0, 0, 1000, 1000]`, todos finitos.


In [ ]:
extreme_logits = np.array([1000.0, -1000.0, 1000.0, -1000.0])
extreme_targets = np.array([1.0, 0.0, 0.0, 1.0])
stable_extreme = bce_with_logits(extreme_logits, extreme_targets, "none")

with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
    p_extreme = sigmoid_stable(extreme_logits)
    naive_extreme = -(
        extreme_targets * np.log(p_extreme)
        + (1.0 - extreme_targets) * np.log(1.0 - p_extreme)
    )

print("probabilidades arredondadas:", p_extreme)
print("BCE ingênua:", naive_extreme)
print("BCE estável:", stable_extreme)


### 9. Clipping muda o objetivo

Clipping com $\epsilon=10^{-12}$ limita a maior penalidade a aproximadamente
$-\log(\epsilon)=27{,}63$. A BCE estável mantém a penalidade correta de cerca de 1000.


In [ ]:
epsilon = 1e-12
wrong_logit = np.array([1000.0])
wrong_target = np.array([0.0])
wrong_probability = np.clip(sigmoid_stable(wrong_logit), epsilon, 1.0 - epsilon)
clipped_loss = bce_from_probabilities(wrong_probability, wrong_target)
stable_wrong_loss = bce_with_logits(wrong_logit, wrong_target)

print({
    "BCE_com_clipping": clipped_loss,
    "BCE_estavel": stable_wrong_loss,
    "penalidade_ocultada": stable_wrong_loss - clipped_loss,
})


### 10. Média ponderada

Pesos são normalizados por sua soma. `mean(weights * losses)` só seria equivalente se
a média dos pesos fosse exatamente um por construção.


In [ ]:
def weighted_mean_loss(values, weights):
    values, weights = same_shape(values, weights, names=("loss", "weights"))
    if np.any(weights < 0.0):
        raise LossContractError("pesos devem ser não negativos")
    weight_sum = float(np.sum(weights))
    if weight_sum <= 0.0:
        raise LossContractError("a soma dos pesos deve ser positiva")
    return float(np.sum(weights * values) / weight_sum)


loss_values = np.array([1.0, 2.0])
weights = np.array([1.0, 3.0])
weighted_correct = weighted_mean_loss(loss_values, weights)
weighted_wrong = float(np.mean(weights * loss_values))

print({"média_ponderada": weighted_correct, "mean(w*l)": weighted_wrong})


### 11. Média global versus média das médias

Para lotes desiguais, agregamos numerador e denominador. Assim o resultado não depende
de onde o vetor foi particionado.


In [ ]:
batch_a = np.array([1.0, 1.0])
batch_b = np.array([9.0])

global_mean = float((batch_a.sum() + batch_b.sum()) /
                    (batch_a.size + batch_b.size))
mean_of_means = float(np.mean([batch_a.mean(), batch_b.mean()]))

all_losses = np.concatenate([batch_a, batch_b])
cuts = [all_losses[:1], all_losses[1:]]
reaggregated = float(sum(part.sum() for part in cuts) /
                     sum(part.size for part in cuts))

print({
    "média_global": global_mean,
    "média_das_médias": mean_of_means,
    "reagregada_outro_corte": reaggregated,
})


### 12. Curvas das losses

O painel esquerdo mostra o crescimento quadrático do MSE com o resíduo. O direito
mostra BCE para alvos 0 e 1 em função do logit: a loss cresce quase linearmente quando
a confiança aponta para a classe errada.


In [ ]:
residual_grid = np.linspace(-5.0, 5.0, 401)
logit_grid = np.linspace(-10.0, 10.0, 401)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(residual_grid, residual_grid**2, color="#386cb0", linewidth=2)
axes[0].set(title="MSE por elemento", xlabel="resíduo (previsão - alvo)", ylabel="loss")
axes[0].grid(alpha=0.25)

axes[1].plot(logit_grid, bce_with_logits(logit_grid, np.zeros_like(logit_grid), "none"),
             label="alvo 0", linewidth=2)
axes[1].plot(logit_grid, bce_with_logits(logit_grid, np.ones_like(logit_grid), "none"),
             label="alvo 1", linewidth=2)
axes[1].set(title="BCE estável", xlabel="logit", ylabel="loss")
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

print("Texto alternativo: à esquerda, parábola simétrica com mínimo zero; "
      "à direita, duas curvas BCE espelhadas que crescem no logit confiante errado.")


## Checks

Os testes abaixo são contratos executáveis. Eles verificam valores manuais, domínios,
reduções, extremos, invariância ao particionamento e falhas intencionais de shape.


In [ ]:
assert np.array_equal(residuals, np.array([1.0, -2.0, 0.0]))
assert np.array_equal(squared, np.array([1.0, 4.0, 0.0]))
assert np.isclose(mse_value, 5.0 / 3.0, atol=1e-15)
assert mse(y_reg, y_reg) == 0.0
assert np.isclose(sum_multi, mean_multi * elementwise_multi.size)

assert np.isclose(bce_manual[0], np.log(2.0), atol=1e-15)
assert np.isclose(bce_manual[1], bce_manual[2], atol=1e-15)
assert np.isclose(bce_manual.mean(), 0.3156677342152968, atol=1e-15)
assert equivalence_error < 2e-10
assert np.isfinite(stable_extreme).all()
assert np.allclose(stable_extreme, np.array([0.0, 0.0, 1000.0, 1000.0]))
assert not np.isfinite(naive_extreme).all()
assert np.isclose(stable_wrong_loss, 1000.0)
assert clipped_loss < 28.0

assert np.isclose(weighted_correct, 1.75)
assert np.isclose(weighted_wrong, 3.5)
assert np.isclose(global_mean, 11.0 / 3.0)
assert np.isclose(mean_of_means, 5.0)
assert np.isclose(reaggregated, global_mean)

try:
    mse(np.zeros((4, 1)), np.zeros(4))
except LossContractError:
    pass
else:
    raise AssertionError("broadcasting perigoso deveria ser rejeitado")

try:
    bce_with_logits(np.array([0.0]), np.array([1.1]))
except LossContractError:
    pass
else:
    raise AssertionError("alvo fora de [0, 1] deveria ser rejeitado")

print("21 contratos verificados com sucesso.")


## Resultados confirmados

- MSE manual: `1.666667`.
- Inserir um erro 10 elevou o MSE de `1.0` para `20.8` e o MAE de `1.0` para `2.8`.
- BCE do exemplo `[0, 2, -2]`: `0.315667734`.
- A forma estável permaneceu finita em logits $\pm1000$; a ingênua produziu não finitos.
- Clipping em $10^{-12}$ reduziu artificialmente a penalidade de `1000` para cerca de `27.631`.
- Média global: `3.666667`; média ingênua das médias: `5.0`.


## Next Steps

Na Aula 07, ampliaremos a classificação para várias classes com softmax e cross-entropy.
Derivaremos log-sum-exp e o gradiente compacto $\mathbf{p}-\mathbf{y}$, ainda em NumPy puro.
